# HT vs. MT Binary Classifier — Softcatalà Spanish-Catalan Corpus

## Overview
Fine-tune `PlanTL-GOB-ES/roberta-base-ca` as a binary classifier to distinguish:
- **Label 1 = HT** — human translation (`.ca` column from Softcatalà)
- **Label 0 = MT** — machine translation of the same Spanish source

## Training corpora (Softcatalà parallel corpus)
| Corpus | Domain |
|--------|--------|
| TildeMODEL | General |
| DOGC | Official / Legal |
| Europarl | Parliamentary |
| open-source | Mixed |

## MT systems used for negative examples
| System | Share |
|--------|-------|
| `Helsinki-NLP/opus-mt-es-ca` | 50% |
| `facebook/nllb-200-distilled-600M` | 50% |

## Output
`translationese_reward(texts) → P(HT | text) ∈ [0, 1]`

Published as: `guerreropaula/ht_mt_classifier_best`


---
## 1. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers datasets evaluate accelerate sentencepiece scikit-learn seaborn openpyxl pandas matplotlib

In [ ]:
import os, subprocess
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Device: {'GPU' if DEVICE == 0 else 'CPU'}")

---
## 2. Download Softcatalà Parallel Corpus

In [ ]:
BASE_URL = "https://github.com/Softcatala/parallel-catalan-corpus/raw/master/spa-cat/"

CORPORA = [
    "TildeMODEL.es-ca",
    "dogc-es-ca",
    "europarl.es-ca",
    "open-source-es-ca",
]

os.makedirs("data/raw", exist_ok=True)

for corpus in CORPORA:
    for lang in ["es", "ca"]:
        fname = f"{corpus}.{lang}"
        out   = f"data/raw/{fname}"
        if not os.path.exists(out):
            for ext in [".xz", ""]:
                url = BASE_URL + fname + ext
                r   = subprocess.run(["wget", "-q", "-O", out + ext, url])
                if r.returncode == 0 and ext == ".xz":
                    subprocess.run(["xz", "-d", out + ".xz"])
                    print(f"  Downloaded & decompressed: {fname}")
                    break
        else:
            print(f"  Already exists: {fname}")

---
## 3. Load, Filter, and Balance Corpora

In [ ]:
frames = []
MAX_PER_CORPUS = 20_000

for corpus in CORPORA:
    es_path = f"data/raw/{corpus}.es"
    ca_path = f"data/raw/{corpus}.ca"
    if os.path.exists(es_path) and os.path.exists(ca_path):
        es_lines = open(es_path, encoding="utf-8").read().splitlines()
        ca_lines = open(ca_path, encoding="utf-8").read().splitlines()
        n = min(len(es_lines), len(ca_lines))
        df = pd.DataFrame({"source_es": es_lines[:n], "ca_human": ca_lines[:n], "corpus": corpus})
        frames.append(df)
        print(f"  {corpus}: {n:,} pairs")

df_all = pd.concat(frames, ignore_index=True)

# Quality filters
df_all = df_all[
    (df_all.source_es.str.len() > 20) & (df_all.ca_human.str.len() > 20) &
    (df_all.source_es.str.len() < 500) & (df_all.ca_human.str.len() < 500)
]

# Balanced sampling
df_balanced = (
    df_all.groupby("corpus", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), MAX_PER_CORPUS), random_state=42))
          .reset_index(drop=True)
          .sample(frac=1, random_state=42)
          .reset_index(drop=True)
)
print(f"\nBalanced dataset: {len(df_balanced):,} pairs")
print(df_balanced.corpus.value_counts())

---
## 4. Generate MT Negative Examples (Helsinki-NLP + NLLB)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer, AutoTokenizer, AutoModelForSeq2SeqLM

HELSINKI_MODEL = "Helsinki-NLP/opus-mt-es-ca"
NLLB_MODEL     = "facebook/nllb-200-distilled-600M"

print("Loading Helsinki-NLP/opus-mt-es-ca...")
hel_tok   = MarianTokenizer.from_pretrained(HELSINKI_MODEL)
hel_model = MarianMTModel.from_pretrained(HELSINKI_MODEL)
if DEVICE == 0: hel_model = hel_model.cuda()
hel_model.eval()

print("Loading facebook/nllb-200-distilled-600M...")
nllb_tok   = AutoTokenizer.from_pretrained(NLLB_MODEL)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL)
if DEVICE == 0: nllb_model = nllb_model.cuda()
nllb_model.eval()
CAT_TOKEN_ID = nllb_tok.convert_tokens_to_ids("cat_Latn")
print("MT models ready.")

In [ ]:
@torch.no_grad()
def batch_translate_helsinki(texts, batch_size=64):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Helsinki"):
        batch = texts[i:i+batch_size]
        enc   = hel_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        if DEVICE == 0: enc = {k: v.cuda() for k, v in enc.items()}
        out = hel_model.generate(**enc, max_length=256)
        results.extend(hel_tok.batch_decode(out, skip_special_tokens=True))
    return results

@torch.no_grad()
def batch_translate_nllb(texts, batch_size=32):
    results = []
    nllb_tok.src_lang = "spa_Latn"
    for i in tqdm(range(0, len(texts), batch_size), desc="NLLB"):
        batch = texts[i:i+batch_size]
        enc   = nllb_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        if DEVICE == 0: enc = {k: v.cuda() for k, v in enc.items()}
        out = nllb_model.generate(**enc, max_length=256, forced_bos_token_id=CAT_TOKEN_ID)
        results.extend(nllb_tok.batch_decode(out, skip_special_tokens=True))
    return results

# Randomly assign 50/50 to Helsinki / NLLB
np.random.seed(42)
assignment   = np.random.choice(["Helsinki", "NLLB"], size=len(df_balanced), p=[0.5, 0.5])
hel_idx      = np.where(assignment == "Helsinki")[0]
nllb_idx     = np.where(assignment == "NLLB")[0]
mt_translations = [""] * len(df_balanced)

print("Translating with Helsinki...")
hel_results = batch_translate_helsinki(df_balanced.iloc[hel_idx]["source_es"].tolist())
for i, t in zip(hel_idx, hel_results): mt_translations[i] = t

print("Translating with NLLB...")
nllb_results = batch_translate_nllb(df_balanced.iloc[nllb_idx]["source_es"].tolist())
for i, t in zip(nllb_idx, nllb_results): mt_translations[i] = t

df_balanced["ca_mt"]     = mt_translations
df_balanced["mt_system"] = assignment
df_balanced.to_csv("df_balanced_mt_ht.csv", index=False)
print("Saved: df_balanced_mt_ht.csv")

import gc
del hel_model, hel_tok, nllb_model, nllb_tok
torch.cuda.empty_cache(); gc.collect()

---
## 5. Build HT vs. MT Classification Dataset

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# HT rows (label = 1)
ht_rows = df_balanced[["ca_human", "corpus"]].copy()
ht_rows.columns = ["text", "corpus"]
ht_rows["label"] = 1

# MT rows (label = 0)
mt_rows = df_balanced[["ca_mt", "corpus", "mt_system"]].copy()
mt_rows.columns = ["text", "corpus", "source"]
mt_rows["label"] = 0

df_clf = pd.concat([ht_rows, mt_rows], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Classifier dataset: {len(df_clf):,} examples")
print(df_clf.label.value_counts())

train_df, val_df = train_test_split(df_clf, test_size=0.1, stratify=df_clf["label"], random_state=42)
train_ds = Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["text","label"]].reset_index(drop=True))
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

---
## 6. Tokenise with `roberta-base-ca`

In [ ]:
from transformers import AutoTokenizer as _AT

MODEL_NAME = "PlanTL-GOB-ES/roberta-base-ca"
clf_tok    = _AT.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = clf_tok(batch["text"], truncation=True, max_length=128, padding="max_length")
    enc["labels"] = batch["label"]
    return enc

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text","label"])
val_tok   = val_ds.map(tokenize,   batched=True, remove_columns=["text","label"])
train_tok.set_format("torch")
val_tok.set_format("torch")
print("Tokenisation complete. Features:", list(train_tok.features.keys()))

---
## 7. Train Classifier

In [ ]:
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, DataCollatorWithPadding)
import evaluate

clf_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0:"MT", 1:"HT"}, label2id={"MT":0,"HT":1}
)

acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {**acc.compute(predictions=preds, references=labels),
            **f1.compute(predictions=preds, references=labels)}

os.makedirs("./ht_mt_classifier", exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = "./ht_mt_classifier",
    num_train_epochs            = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 32,
    warmup_ratio                = 0.06,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

trainer = Trainer(
    model           = clf_model,
    args            = training_args,
    train_dataset   = train_tok,
    eval_dataset    = val_tok,
    data_collator   = DataCollatorWithPadding(clf_tok),
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()
print("\nValidation results:")
for k, v in trainer.evaluate().items():
    print(f"  {k}: {v:.4f}")

---
## 8. Save and Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

CLF_OUTPUT_DIR = "./ht_mt_classifier_best"
trainer.save_model(CLF_OUTPUT_DIR)
clf_tok.save_pretrained(CLF_OUTPUT_DIR)
print(f"Saved to {CLF_OUTPUT_DIR}")

HF_TOKEN = ""  # ← set your token
login(token=HF_TOKEN) if HF_TOKEN else login()

trainer.push_to_hub("guerreropaula/ht_mt_classifier_best")
clf_tok.push_to_hub("guerreropaula/ht_mt_classifier_best")
print("Pushed to https://huggingface.co/guerreropaula/ht_mt_classifier_best")

---
## 9. GRPO Integration — `translationese_reward`

Once trained, wrap the classifier as a reward function for the GRPO training loop.


In [ ]:
import torch.nn.functional as F
from transformers import AutoTokenizer as _AT2, AutoModelForSequenceClassification as _AMSC

CLF_REPO_ID  = "guerreropaula/ht_mt_classifier_best"
HT_LABEL_IDX = 1

_clf_tok2   = _AT2.from_pretrained(CLF_REPO_ID)
_clf_model2 = _AMSC.from_pretrained(CLF_REPO_ID).eval()
if torch.cuda.is_available():
    _clf_model2 = _clf_model2.cuda()

@torch.no_grad()
def translationese_reward(texts, batch_size=16):
    """Returns P(HT | text) ∈ [0, 1] for each text. Use as r_t in GRPO."""
    rewards = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = _clf_tok2(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=256)
        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}
        probs = F.softmax(_clf_model2(**enc).logits, dim=-1)
        rewards.extend(probs[:, HT_LABEL_IDX].cpu().tolist())
    return rewards

# Quick test
test_texts = [
    "L'ajuntament ha aprovat el nou pressupost.",
    "The town hall has approved the new budget.",
    "El ayuntamiento ha aprobado el nuevo presupuesto.",
]
scores = translationese_reward(test_texts)
for t, s in zip(test_texts, scores):
    print(f"  P(HT)={s:.3f}  |  {t[:60]}")